# 05 — Clustering: perfiles de personal ESPOL

**Objetivo:** encontrar agrupaciones de personas con características similares a partir de
`data/modeling/X_modelado.csv`, evaluando distintos algoritmos y números de clusters, y
seleccionar una solución final interpretable y útil para el objetivo de la tesis
(apoyo a la asignación de tareas, conformación de comisiones/equipos, planificación).

**Entradas (ya preparadas en la etapa 04, no se repiten aquí):**

- `data/modeling/X_modelado.csv` — matriz numérica lista para modelar (2213 × 100), sin nulos ni infinitos.
- `data/modeling/personas_modelado.csv` — relación fila ↔ `IDPERSONA`.
- `data/modeling/feature_names_modelado.csv` — nombres de las 100 columnas finales y su origen.
- `data/modeling/preprocessing_summary.csv` — resumen de imputación/transformación/escalamiento por variable original.
- `data/features/dataset_personas_features.csv` — features **originales/interpretables** (85 variables), usadas
  únicamente para caracterizar los clusters, nunca para el clustering en sí.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
)
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

ROOT = Path.cwd().parents[1] if (Path.cwd().name == "05_clustering") else Path.cwd()
DATA_MODELING = ROOT / "data" / "modeling"
DATA_FEATURES = ROOT / "data" / "features"
DATA_CLUSTERING = ROOT / "data" / "clustering"
DATA_CLUSTERING.mkdir(parents=True, exist_ok=True)

print("Directorio de trabajo:", ROOT)
print("Salidas de clustering en:", DATA_CLUSTERING)


## 1. Carga y verificación de datos

In [ ]:
X_df = pd.read_csv(DATA_MODELING / "X_modelado.csv")
personas = pd.read_csv(DATA_MODELING / "personas_modelado.csv")
feature_names = pd.read_csv(DATA_MODELING / "feature_names_modelado.csv")
preprocessing_summary = pd.read_csv(DATA_MODELING / "preprocessing_summary.csv")

print("X_modelado:", X_df.shape)
print("personas_modelado:", personas.shape)
print("feature_names_modelado:", feature_names.shape)


In [ ]:
# Verificaciones básicas de integridad
n_nulos = int(X_df.isnull().sum().sum())
n_infinitos = int(np.isinf(X_df.to_numpy()).sum())

assert X_df.shape[0] == personas.shape[0], "Filas de X_modelado y personas_modelado no coinciden"
assert X_df.shape[1] == feature_names.shape[0], "Número de columnas no coincide con feature_names_modelado"
assert list(personas["INDICE_X_MODELADO"]) == list(range(len(personas))), \
    "El índice de personas_modelado no corresponde 1:1 con las filas de X_modelado"
assert personas["IDPERSONA"].is_unique, "Hay IDPERSONA duplicados en personas_modelado"

print(f"Personas               : {X_df.shape[0]}")
print(f"Features               : {X_df.shape[1]}")
print(f"Nulos totales          : {n_nulos}")
print(f"Infinitos totales      : {n_infinitos}")
print(f"IDPERSONA únicos       : {personas['IDPERSONA'].nunique()}")
print(f"Correspondencia fila<->IDPERSONA verificada: OK")


In [ ]:
# Distribución general de la matriz (ya escalada/codificada en la etapa 04)
X = X_df.to_numpy()

resumen_valores = pd.Series(X.ravel()).describe()
print(resumen_valores)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(X.ravel(), bins=80, color="#4C72B0")
axes[0].set_title("Distribución de todos los valores de X_modelado")
axes[0].set_xlabel("valor")
axes[0].set_ylabel("frecuencia")

col_std = X_df.std().sort_values(ascending=False)
axes[1].hist(col_std, bins=30, color="#55A868")
axes[1].set_title("Desviación estándar por columna")
axes[1].set_xlabel("std")
axes[1].set_ylabel("num. columnas")

plt.tight_layout()
plt.show()


**Nota sobre la naturaleza mixta de `X_modelado`:** según `preprocessing_summary.csv`, las 72
variables numéricas (y la ordinal `NIVEL_ACADEMICO_MAXIMO_ORD`) fueron estandarizadas con
`StandardScaler` (media 0, varianza 1), mientras que las variables categóricas codificadas con
one-hot (`nom__...`) y las binarias (`bin__...`) quedaron en escala 0/1 **sin estandarizar**. Esto
es una decisión ya tomada en la etapa 04 y no se modifica aquí, pero implica que, en un clustering
basado en distancia euclídea (K-Means, jerárquico), las variables numéricas estandarizadas
dominan más la distancia que las variables dummy 0/1. Se documenta como una característica/limitación
conocida de la representación utilizada, relevante al interpretar los resultados.


## 2. Selección del número de clusters (K) — K-Means

Se evalúan varios valores de K con K-Means (`n_init=10`, `random_state` fijo para esta primera
pasada) y se registran métricas de calidad y tamaños de cluster. La estabilidad ante distintas
semillas se evalúa por separado en la sección 3.


In [ ]:
K_RANGE = range(2, 13)

filas_k = []
labels_por_k = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)
    labels_por_k[k] = labels
    sizes = pd.Series(labels).value_counts()
    filas_k.append({
        "K": k,
        "SILHOUETTE": silhouette_score(X, labels),
        "CALINSKI_HARABASZ": calinski_harabasz_score(X, labels),
        "DAVIES_BOULDIN": davies_bouldin_score(X, labels),
        "MIN_CLUSTER_SIZE": int(sizes.min()),
        "MAX_CLUSTER_SIZE": int(sizes.max()),
    })

evaluacion_k = pd.DataFrame(filas_k)
evaluacion_k.to_csv(DATA_CLUSTERING / "evaluacion_k.csv", index=False)
evaluacion_k


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(evaluacion_k["K"], evaluacion_k["SILHOUETTE"], marker="o")
axes[0, 0].set_title("Silhouette (mayor es mejor)")
axes[0, 0].set_xlabel("K")

axes[0, 1].plot(evaluacion_k["K"], evaluacion_k["CALINSKI_HARABASZ"], marker="o", color="#55A868")
axes[0, 1].set_title("Calinski-Harabasz (mayor es mejor)")
axes[0, 1].set_xlabel("K")

axes[1, 0].plot(evaluacion_k["K"], evaluacion_k["DAVIES_BOULDIN"], marker="o", color="#C44E52")
axes[1, 0].set_title("Davies-Bouldin (menor es mejor)")
axes[1, 0].set_xlabel("K")

axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MIN_CLUSTER_SIZE"], marker="o", label="mínimo")
axes[1, 1].plot(evaluacion_k["K"], evaluacion_k["MAX_CLUSTER_SIZE"], marker="o", label="máximo")
axes[1, 1].set_title("Tamaño de clusters")
axes[1, 1].set_xlabel("K")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


**Lectura de las métricas de K-Means:**

- El **Silhouette** es máximo en `K=2` y decrece de forma monótona después. Esto es esperable en una
  matriz de 100 dimensiones con variables dummy: una partición muy gruesa maximiza la separación
  promedio, pero no necesariamente aporta perfiles útiles para la tesis (ver más abajo qué separa
  a K=2).
- **Calinski-Harabasz** sigue el mismo patrón decreciente — también favorece soluciones gruesas y no
  es, por sí solo, un criterio suficiente.
- **Davies-Bouldin** (menor es mejor) es más plano entre K=4 y K=9, sin un mínimo claramente dominante.
- Los **tamaños de cluster** se mantienen razonablemente balanceados (ningún cluster
  extremadamente pequeño) hasta aproximadamente K=7-8; a partir de K=9 empiezan a aparecer clusters
  de menos de 100 personas.

Ninguna métrica por sí sola debe decidir K (así lo indica también la consigna del proyecto). Antes
de decidir, se revisa qué representa realmente la partición en K=2, y se evalúa la **estabilidad**
de las particiones (sección 3) y su **interpretabilidad** (sección 6/7).


In [ ]:
# ¿Qué separa a la partición más gruesa (K=2)?
feat_orig = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")

tmp = personas[["IDPERSONA"]].copy()
tmp["CLUSTER_K2"] = labels_por_k[2]
tmp = tmp.merge(feat_orig[["IDPERSONA", "TIPOEMPLEADO_ACTUAL_DESC"]], on="IDPERSONA", how="left")

print(pd.crosstab(tmp["CLUSTER_K2"], tmp["TIPOEMPLEADO_ACTUAL_DESC"]))


K=2 no separa de forma limpia personal docente de administrativo (ambos tipos aparecen en los dos
clusters); el corte principal está dominado por el volumen general de actividad/antigüedad. Es una
partición demasiado agregada para el objetivo de la tesis (perfiles multidimensionales útiles para
asignación de tareas), por lo que **no se selecciona únicamente por tener el mayor Silhouette**.


## 3. Estabilidad de K-Means

K-Means depende de la inicialización aleatoria. Para cada K se ejecuta el algoritmo con varias
semillas distintas y se mide el **Adjusted Rand Index (ARI)** entre cada par de particiones
resultantes. Un ARI cercano a 1 indica que, independientemente de la semilla, el algoritmo converge
esencialmente a la misma partición (solución estable); valores bajos indican que la solución depende
demasiado del azar de inicialización para ese K.


In [ ]:
SEEDS = [0, 1, 2, 3, 4, 42, 100, 123]

filas_estab = []
for k in K_RANGE:
    labelings = [
        KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X)
        for s in SEEDS
    ]
    aris = [
        adjusted_rand_score(labelings[i], labelings[j])
        for i in range(len(labelings))
        for j in range(i + 1, len(labelings))
    ]
    aris = np.array(aris)
    filas_estab.append({
        "K": k,
        "ARI_MEAN": aris.mean(),
        "ARI_STD": aris.std(),
        "ARI_MIN": aris.min(),
    })

estabilidad_k = pd.DataFrame(filas_estab)
estabilidad_k


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.errorbar(
    estabilidad_k["K"], estabilidad_k["ARI_MEAN"], yerr=estabilidad_k["ARI_STD"],
    marker="o", capsize=3,
)
ax.axhline(0.9, color="gray", linestyle="--", linewidth=1, label="ARI = 0.90")
ax.set_xlabel("K")
ax.set_ylabel("ARI promedio entre semillas")
ax.set_title("Estabilidad de K-Means ante distintas semillas")
ax.legend()
plt.tight_layout()
plt.show()


**Lectura de estabilidad:** las particiones son muy estables (ARI ≥ 0.94) entre K=2 y K=7, con un
máximo local notable en **K=5** (ARI promedio ≈ 0.99). A partir de K=8 la estabilidad cae de forma
apreciable (ARI < 0.87 y con mayor dispersión), señal de que en ese rango K-Means empieza a producir
particiones distintas según la inicialización — es decir, la estructura de los datos ya no sostiene
con claridad ese número de grupos. Esto descarta, en la práctica, valores de K ≥ 8 para la solución
final, y refuerza a K=5 como un punto interesante dentro del rango estable.


## 4. Clustering jerárquico (Agglomerative)

Como segundo enfoque se evalúa clustering aglomerativo con enlace `ward` (el más comparable a
K-Means porque también minimiza varianza intra-cluster) y distancia euclídea, para el mismo rango
de K identificado como razonable (K=3 a K=7). No se prueban decenas de combinaciones de
enlace/distancia — solo configuraciones razonables — y el dendrograma se usa únicamente como
herramienta exploratoria, no como criterio de corte definitivo.


In [ ]:
# Dendrograma exploratorio (truncado; ward/euclídea sobre X completo)
Z = linkage(X, method="ward")

fig, ax = plt.subplots(figsize=(10, 5))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True, ax=ax)
ax.set_title("Dendrograma (ward, truncado a los últimos 30 nodos) — solo exploratorio")
ax.set_xlabel("tamaño de cluster (o índice) en cada nodo")
ax.set_ylabel("distancia (ward)")
plt.tight_layout()
plt.show()


In [ ]:
filas_agg = []
labels_agg_por_k = {}

for k in [3, 4, 5, 6, 7]:
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels_agg = agg.fit_predict(X)
    labels_agg_por_k[k] = labels_agg
    sizes = pd.Series(labels_agg).value_counts()
    ari_vs_kmeans = adjusted_rand_score(labels_por_k[k], labels_agg)
    filas_agg.append({
        "K": k,
        "SILHOUETTE": silhouette_score(X, labels_agg),
        "CALINSKI_HARABASZ": calinski_harabasz_score(X, labels_agg),
        "DAVIES_BOULDIN": davies_bouldin_score(X, labels_agg),
        "MIN_CLUSTER_SIZE": int(sizes.min()),
        "MAX_CLUSTER_SIZE": int(sizes.max()),
        "ARI_VS_KMEANS": ari_vs_kmeans,
    })

comparacion_agg = pd.DataFrame(filas_agg)
comparacion_agg


**Comparación jerárquico vs. K-Means:** para todos los K evaluados, K-Means iguala o supera a
Agglomerative(ward) en Silhouette, Calinski-Harabasz y Davies-Bouldin. El ARI entre ambas
soluciones para el mismo K se mantiene entre ~0.5 y ~0.66 — es decir, **coinciden de forma
sustancial pero no idéntica**: hay acuerdo estructural razonable entre dos algoritmos con criterios
de optimización distintos, lo que da algo de confianza en que la estructura detectada no es un
artefacto de un solo algoritmo. K-Means se usa como referencia principal por su mejor desempeño en
las tres métricas; el jerárquico se usa como validación cruzada del enfoque, no como candidato
final independiente.


## 5. DBSCAN / HDBSCAN

Se evalúan métodos basados en densidad **únicamente si tienen sentido para esta matriz**: 100
dimensiones, con una mezcla de variables numéricas estandarizadas y variables dummy 0/1 dispersas
(one-hot y binarias). En alta dimensionalidad, las distancias tienden a concentrarse y la densidad
deja de ser un concepto bien definido ("curse of dimensionality"), por lo que antes de forzar el uso
de estos métodos se hace una verificación empírica rápida.


In [ ]:
# Distancia al 10º vecino más cercano, para orientar la elección de eps en DBSCAN
nn = NearestNeighbors(n_neighbors=10).fit(X)
dist_10nn, _ = nn.kneighbors(X)
k_dist = np.sort(dist_10nn[:, -1])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_dist)
ax.set_title("Distancia al 10º vecino más cercano (ordenada) — orientación para eps de DBSCAN")
ax.set_xlabel("puntos, ordenados por distancia")
ax.set_ylabel("distancia al 10º vecino")
plt.tight_layout()
plt.show()

print(pd.Series(k_dist).describe())


In [ ]:
filas_dbscan = []
for eps in [3, 4, 5, 6, 8, 10]:
    for min_samples in [5, 10]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = int((labels == -1).sum())
        filas_dbscan.append({
            "EPS": eps, "MIN_SAMPLES": min_samples,
            "N_CLUSTERS": n_clusters, "N_NOISE": n_noise,
            "PCT_NOISE": round(100 * n_noise / len(labels), 1),
        })

dbscan_trials = pd.DataFrame(filas_dbscan)
dbscan_trials


In [ ]:
filas_hdbscan = []
for mcs in [20, 30, 50, 80, 100]:
    labels = HDBSCAN(min_cluster_size=mcs).fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = int((labels == -1).sum())
    filas_hdbscan.append({
        "MIN_CLUSTER_SIZE": mcs,
        "N_CLUSTERS": n_clusters, "N_NOISE": n_noise,
        "PCT_NOISE": round(100 * n_noise / len(labels), 1),
    })

hdbscan_trials = pd.DataFrame(filas_hdbscan)
hdbscan_trials


**Conclusión DBSCAN/HDBSCAN — no se usan como solución final:**

- Con `eps` bajo (3-4), DBSCAN deja entre 75% y 93% de las personas como *ruido*, formando solo 2-5
  clusters diminutos.
- Al aumentar `eps` (8-10) para reducir el ruido, el algoritmo colapsa a **un único cluster gigante**
  que contiene prácticamente a todos (0.9%-9.8% de ruido, 1 solo cluster) — no hay un rango
  intermedio de `eps` que produzca varios clusters sustanciales con poco ruido.
- HDBSCAN muestra el mismo patrón: con `min_cluster_size` bajo encuentra 2 clusters muy pequeños
  (decenas de personas) dejando >90% como ruido; con `min_cluster_size` ≥ 50 no encuentra ningún
  cluster (100% ruido).

Este comportamiento es el esperado en una matriz de 100 dimensiones con variables dummy dispersas:
no existe una noción de densidad bien diferenciada que separe grupos densos de regiones vacías. Tal
como indica la consigna del proyecto, **se documenta esta limitación y no se fuerza el uso de
métodos basados en densidad** para la solución final; K-Means y el jerárquico (ambos basados en
distancia/varianza, no en densidad) son más adecuados para esta representación.


## 6. Selección del modelo final

Se resumen los criterios evaluados para los K candidatos (aquellos con estabilidad alta, K=3 a K=7)
bajo K-Means, y se comparan cualitativamente contra el jerárquico y contra la partición trivial K=2.

Criterios considerados conjuntamente (ninguno de forma aislada):

1. **Calidad estadística** (Silhouette, Calinski-Harabasz, Davies-Bouldin).
2. **Estabilidad** (ARI promedio entre semillas, sección 3).
3. **Tamaños razonables de cluster** (sin grupos extremadamente pequeños).
4. **Acuerdo con un algoritmo independiente** (ARI vs. Agglomerative, sección 4).
5. **Interpretabilidad / utilidad para el objetivo de la tesis** (se verifica a continuación con las
   features originales, antes de decidir).


In [ ]:
resumen_candidatos = (
    evaluacion_k[evaluacion_k["K"].between(3, 7)]
    .merge(estabilidad_k, on="K")
    .merge(comparacion_agg[["K", "ARI_VS_KMEANS"]], on="K")
)
resumen_candidatos


In [ ]:
# Vista previa de interpretabilidad: mediana de variables clave por cluster, para cada K candidato
feat_orig_idx = personas[["IDPERSONA"]].merge(feat_orig, on="IDPERSONA", how="left")

cols_preview = [
    "TOTAL_HORAS_DOCENCIA", "NUM_PUBLICACIONES", "NUM_PROYECTOS_INVESTIGACION",
    "ANIOS_EXPERIENCIA_DOCENTE", "ANIOS_EXPERIENCIA_ADMINISTRATIVO", "ANTIGUEDAD_EFECTIVA_ANIOS",
]

for k in [3, 4, 5, 6]:
    df_prev = feat_orig_idx.copy()
    df_prev["CLUSTER"] = labels_por_k[k]
    print(f"--- K={k} (tamaños: {df_prev['CLUSTER'].value_counts().sort_index().to_dict()}) ---")
    print(df_prev.groupby("CLUSTER")[cols_preview].median().round(2))
    print()


**Decisión — K-Means con K=5:**

- **Estabilidad:** K=5 tiene la estabilidad más alta de todo el rango evaluado (ARI promedio ≈ 0.99
  entre semillas), superando incluso a K=3 y K=4.
- **Calidad estadística:** dentro del rango K=3-7 (una vez descartado K=2 por ser demasiado grueso),
  K=5 mantiene un Silhouette y Davies-Bouldin comparables a sus vecinos, sin ser el peor en ninguna
  métrica.
- **Acuerdo con el jerárquico:** K=5 es el K con mayor ARI frente a Agglomerative(ward) (~0.66) de
  todo el rango candidato, lo que indica que la partición no depende de las particularidades de
  K-Means.
- **Tamaños de cluster:** los 5 grupos quedan entre ~260 y ~590 personas (11.8%-26.6% de la
  población) — ningún grupo extremadamente pequeño ni dominante.
- **Interpretabilidad:** al revisar las medianas de variables originales (celda anterior), K=5 es el
  primer K que separa con claridad **cinco** patrones distintos y sustantivos: un perfil
  administrativo puro, un perfil docente-investigador senior, un perfil de alta carga docente sin
  investigación, un perfil docente de desarrollo/carga moderada, y un perfil de ingreso reciente sin
  trayectoria consolidada. K=3 y K=4 mezclan algunos de estos patrones en un mismo cluster (p. ej. no
  distinguen "ingreso reciente" de "carga docente moderada"), mientras que K=6 y K=7 no aportan un
  patrón adicional claramente distinto — solo fragmentan alguno de los cinco perfiles anteriores sin
  ganancia interpretativa proporcional, y ya muestran menor estabilidad.

Por lo tanto, **no se elige K únicamente por el mejor Silhouette** (que sería K=2) ni se usa un
método basado en densidad (descartado en la sección 5). Se selecciona **K-Means, K=5,
`random_state=42`, `n_init=10`** como solución final, por ser la que mejor combina estabilidad,
calidad estadística competitiva, tamaños balanceados e interpretabilidad sustantiva para el objetivo
de la tesis. Esta decisión debe registrarse en `context/DECISION_LOG.md` como una decisión formal del
proyecto.


In [ ]:
K_FINAL = 5

modelo_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=RANDOM_STATE)
cluster_labels = modelo_final.fit_predict(X)

print("Tamaños de cluster:")
print(pd.Series(cluster_labels).value_counts().sort_index())
print()
print(f"Silhouette         : {silhouette_score(X, cluster_labels):.4f}")
print(f"Calinski-Harabasz  : {calinski_harabasz_score(X, cluster_labels):.2f}")
print(f"Davies-Bouldin     : {davies_bouldin_score(X, cluster_labels):.4f}")


In [ ]:
# data/clustering/clusters_personas.csv — una fila por persona
clusters_personas = personas[["IDPERSONA"]].copy()
clusters_personas["CLUSTER"] = cluster_labels

assert clusters_personas["IDPERSONA"].is_unique
assert len(clusters_personas) == X_df.shape[0]

clusters_personas.to_csv(DATA_CLUSTERING / "clusters_personas.csv", index=False)
clusters_personas.head()


In [ ]:
# Modelo serializado
joblib.dump(modelo_final, DATA_CLUSTERING / "modelo_clustering.joblib")
print("Modelo guardado en", DATA_CLUSTERING / "modelo_clustering.joblib")


In [ ]:
# data/clustering/clustering_metrics.csv — métricas del modelo final seleccionado
ari_estab_final = estabilidad_k.loc[estabilidad_k["K"] == K_FINAL, "ARI_MEAN"].iloc[0]
ari_vs_agg_final = comparacion_agg.loc[comparacion_agg["K"] == K_FINAL, "ARI_VS_KMEANS"].iloc[0]

clustering_metrics = pd.DataFrame([{
    "MODELO": "KMeans",
    "K": K_FINAL,
    "N_PERSONAS": X_df.shape[0],
    "N_FEATURES": X_df.shape[1],
    "SILHOUETTE": silhouette_score(X, cluster_labels),
    "CALINSKI_HARABASZ": calinski_harabasz_score(X, cluster_labels),
    "DAVIES_BOULDIN": davies_bouldin_score(X, cluster_labels),
    "ARI_ESTABILIDAD_SEMILLAS": ari_estab_final,
    "ARI_VS_AGGLOMERATIVE": ari_vs_agg_final,
    "RANDOM_STATE": RANDOM_STATE,
    "N_INIT": 10,
}])
clustering_metrics.to_csv(DATA_CLUSTERING / "clustering_metrics.csv", index=False)
clustering_metrics


In [ ]:
# data/clustering/cluster_sizes.csv
cluster_sizes = (
    clusters_personas["CLUSTER"].value_counts().sort_index()
    .rename_axis("CLUSTER").reset_index(name="N_PERSONAS")
)
cluster_sizes["PCT_POBLACION"] = (100 * cluster_sizes["N_PERSONAS"] / len(clusters_personas)).round(2)
cluster_sizes.to_csv(DATA_CLUSTERING / "cluster_sizes.csv", index=False)
cluster_sizes


## 7. Caracterización de clusters

La caracterización se hace **con las features originales/interpretables** de
`data/features/dataset_personas_features.csv` (85 variables, en su unidad y escala natural),
**no** con las 100 columnas transformadas de `X_modelado.csv` que se usaron para el clustering.
Se relaciona `IDPERSONA` + `CLUSTER` (de `clusters_personas.csv`) con ese dataset.

Cada variable original se trata según su tipo, tomado de `data/features/feature_dictionary.csv`:

- **Numéricas:** mediana por cluster vs. mediana global; `IMPORTANCE` = diferencia estandarizada
  (`|mediana_cluster - mediana_global| / std_global`), un tamaño de efecto simple.
- **Booleanas:** proporción de `True` por cluster vs. proporción global; `IMPORTANCE` = diferencia
  absoluta de proporciones.
- **Categóricas (incluye la ordinal `NIVEL_ACADEMICO_MAXIMO`):** se identifica la categoría más
  frecuente a nivel global y se compara qué proporción de cada cluster cae en esa categoría frente a
  la proporción global; adicionalmente se reporta la categoría más frecuente **dentro** de cada
  cluster (que puede diferir de la global), para la interpretación cualitativa.

`IDPERSONA` se excluye por ser un identificador, no una característica.


In [ ]:
feat_orig = pd.read_csv(DATA_FEATURES / "dataset_personas_features.csv")
feat_dict = pd.read_csv(DATA_FEATURES / "feature_dictionary.csv")

df_carac = clusters_personas.merge(feat_orig, on="IDPERSONA", how="left")
assert df_carac["CLUSTER"].isnull().sum() == 0

tipo_por_feature = dict(zip(feat_dict["FEATURE"], feat_dict["TYPE"]))
features_a_caracterizar = [c for c in feat_orig.columns if c != "IDPERSONA"]

clusters_ordenados = sorted(df_carac["CLUSTER"].unique())
filas_caract = []

for feature in features_a_caracterizar:
    tipo = tipo_por_feature.get(feature, "numerica")
    col = df_carac[feature]

    if tipo == "numerica":
        global_val = col.median()
        global_std = col.std()
        for c in clusters_ordenados:
            val_c = col[df_carac["CLUSTER"] == c].median()
            diff = val_c - global_val
            importance = abs(diff) / global_std if global_std and global_std > 0 else 0.0
            if diff > 0:
                signo = "muy por encima" if importance >= 0.8 else "por encima"
            elif diff < 0:
                signo = "muy por debajo" if importance >= 0.8 else "por debajo"
            else:
                signo = "en línea con"
            interpretacion = f"Mediana de {feature} {signo} del promedio institucional ({val_c:g} vs {global_val:g})."
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))

    elif tipo == "booleana":
        col_bool = col.astype("boolean")
        global_val = col_bool.mean(skipna=True)
        for c in clusters_ordenados:
            val_c = col_bool[df_carac["CLUSTER"] == c].mean(skipna=True)
            diff = val_c - global_val
            importance = abs(diff)
            interpretacion = (
                f"{val_c:.0%} del cluster cumple {feature}, frente a {global_val:.0%} a nivel global."
            )
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))

    elif tipo in ("categorica", "categorica_ordinal"):
        col_str = col.astype("string").fillna("SIN_DATO")
        vc_global = col_str.value_counts(normalize=True)
        global_mode, global_val = vc_global.index[0], vc_global.iloc[0]
        for c in clusters_ordenados:
            sub = col_str[df_carac["CLUSTER"] == c]
            vc_c = sub.value_counts(normalize=True)
            cluster_mode, cluster_mode_prop = vc_c.index[0], vc_c.iloc[0]
            val_c = vc_c.get(global_mode, 0.0)
            diff = val_c - global_val
            importance = abs(diff)
            interpretacion = (
                f"Categoría predominante en el cluster: '{cluster_mode}' ({cluster_mode_prop:.0%}); "
                f"a nivel global la más común es '{global_mode}' ({global_val:.0%})."
            )
            filas_caract.append(dict(
                CLUSTER=c, FEATURE=feature, VALUE_CLUSTER=val_c, VALUE_GLOBAL=global_val,
                DIFFERENCE=diff, IMPORTANCE=importance, INTERPRETACION=interpretacion,
            ))
    # tipo "identificador" (no debería aparecer aquí) se ignora

cluster_characterization = (
    pd.DataFrame(filas_caract)
    .sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .reset_index(drop=True)
)
cluster_characterization.to_csv(DATA_CLUSTERING / "cluster_characterization.csv", index=False)
print(cluster_characterization.shape)
cluster_characterization.head(10)


In [ ]:
# Top 8 variables más diferenciadoras por cluster (para revisión rápida / evidencia de nombres)
for c in clusters_ordenados:
    print(f"\n=== CLUSTER {c} (n={cluster_sizes.loc[cluster_sizes.CLUSTER==c, 'N_PERSONAS'].iloc[0]}) — variables más diferenciadoras ===")
    top = cluster_characterization[cluster_characterization["CLUSTER"] == c].head(8)
    for _, r in top.iterrows():
        print(f"  - {r['INTERPRETACION']}")


In [ ]:
# data/clustering/cluster_profiles.csv — resumen por cluster (formato ancho)
numericas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) == "numerica"]
booleanas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) == "booleana"]
categoricas = [f for f in features_a_caracterizar if tipo_por_feature.get(f) in ("categorica", "categorica_ordinal")]

perfiles = []
n_total = len(df_carac)
for c in clusters_ordenados:
    sub = df_carac[df_carac["CLUSTER"] == c]
    fila = {"CLUSTER": c, "N_PERSONAS": len(sub), "PCT_POBLACION": round(100 * len(sub) / n_total, 2)}
    for f in numericas:
        fila[f"{f}__MEDIA"] = sub[f].mean()
        fila[f"{f}__MEDIANA"] = sub[f].median()
    for f in booleanas:
        fila[f"{f}__PROPORCION"] = sub[f].astype("boolean").mean(skipna=True)
    for f in categoricas:
        vc = sub[f].astype("string").fillna("SIN_DATO").value_counts(normalize=True)
        fila[f"{f}__MODA"] = vc.index[0]
        fila[f"{f}__MODA_PROP"] = vc.iloc[0]
    perfiles.append(fila)

cluster_profiles = pd.DataFrame(perfiles)
cluster_profiles.to_csv(DATA_CLUSTERING / "cluster_profiles.csv", index=False)
print(cluster_profiles.shape)
cluster_profiles[["CLUSTER", "N_PERSONAS", "PCT_POBLACION"]]


## 8. Nombres de los perfiles

Los nombres se proponen **después** de revisar la evidencia de la sección anterior (variables más
diferenciadoras por cluster), no antes. Se resume esa evidencia por cluster:

- **Cluster 0** (~483 personas, 21.8%): docentes con actividad de docencia histórica moderada (mediana
  1233 horas, 4 periodos, 17 cursos), en su mayoría con un solo régimen contractual y nivel académico
  de cuarto nivel — pero con `VIGENTE_ACTUALMENTE` muy por debajo del global (3.7% vs 30.9%): son, en
  su mayoría, personas cuya relación laboral docente ya no está vigente.
  → **"Perfil docente de trayectoria histórica moderada (mayormente no vigente)"**.

- **Cluster 1** (~588 personas, 26.6%): 95.7% administrativos, con la mayor experiencia administrativa
  (mediana 11.15 años vs 0.84 global), régimen LOSEP predominante y mayor proporción actualmente
  vigente (67.3% vs 30.9% global).
  → **"Perfil administrativo"**.

- **Cluster 2** (~262 personas, 11.8%): el grupo con más actividad en casi todas las dimensiones:
  máximo de periodos de docencia, más horas y actividades politécnicas, más proyectos de
  investigación (incl. como director), más publicaciones, alta proporción de roles docente-administrativo
  mixtos (72.5%) y de múltiples cargos en un mismo año (95.4%). Todos con nivel académico de cuarto
  nivel.
  → **"Perfil de alta producción académica e investigativa / liderazgo integral"**.

- **Cluster 3** (~326 personas, 14.7%): el grupo con mayor carga de docencia pura — máximo de horas de
  docencia (mediana 5082 h), más cursos (63) y más estudiantes atendidos (1406), con actividad
  politécnica alta, pero sin el mismo nivel de producción investigativa que el Cluster 2.
  → **"Perfil de alta carga docente"**.

- **Cluster 4** (~554 personas, 25.0%): antigüedad efectiva y calendario muy bajas (mediana ~1.2-1.4
  años), pocos registros históricos (mediana 4 vs 20.5 global), un solo cargo distinto, pocas
  capacitaciones, nivel académico predominante de tercer nivel (por debajo del cuarto nivel
  predominante a nivel global) y prácticamente nadie vigente actualmente (0.5%).
  → **"Perfil de ingreso reciente / trayectoria corta"**.

**Advertencia importante:** estos nombres son una síntesis interpretativa basada en los datos
disponibles, no una clasificación oficial ni una categoría administrativa de ESPOL. No implican un
juicio de valor sobre las personas de cada grupo, y no deben usarse como criterio automático de
decisiones sobre contratación, promoción o asignación sin la intervención y el criterio de las
unidades institucionales correspondientes (ver `context/00-contexto-permanente.md`).


In [ ]:
nombres_perfiles = {
    0: "Perfil docente de trayectoria histórica moderada (mayormente no vigente)",
    1: "Perfil administrativo",
    2: "Perfil de alta producción académica e investigativa / liderazgo integral",
    3: "Perfil de alta carga docente",
    4: "Perfil de ingreso reciente / trayectoria corta",
}

nombres_df = pd.DataFrame({
    "CLUSTER": list(nombres_perfiles.keys()),
    "NOMBRE_PERFIL": list(nombres_perfiles.values()),
})
cluster_sizes_nombrado = cluster_sizes.merge(nombres_df, on="CLUSTER")
cluster_sizes_nombrado


## 9. Visualizaciones

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
orden = cluster_sizes_nombrado.sort_values("CLUSTER")
bars = ax.bar(orden["CLUSTER"].astype(str), orden["N_PERSONAS"], color=sns.color_palette("Set2", len(orden)))
for bar, pct in zip(bars, orden["PCT_POBLACION"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, f"{pct:.1f}%", ha="center")
ax.set_xlabel("Cluster")
ax.set_ylabel("N° de personas")
ax.set_title(f"Tamaño de los {K_FINAL} clusters finales (K-Means)")
plt.tight_layout()
plt.show()


### Visualización 2D (solo para inspección visual — no reemplaza la matriz de clustering)

Se proyecta `X_modelado` a 2 componentes con PCA **únicamente para poder graficar** los clusters en
un plano. El clustering en sí se realizó, y se selecciona, sobre las 100 dimensiones originales de
`X_modelado`; esta proyección 2D es solo una ayuda visual y pierde información — dos personas pueden
verse cercanas en el gráfico y no serlo en el espacio completo, o viceversa.


In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
var_explicada = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 6))
paleta = sns.color_palette("Set2", K_FINAL)
for c in clusters_ordenados:
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=12, alpha=0.6, color=paleta[c], label=f"Cluster {c}")
ax.set_xlabel(f"PC1 ({var_explicada[0]:.1%} var. explicada)")
ax.set_ylabel(f"PC2 ({var_explicada[1]:.1%} var. explicada)")
ax.set_title("Clusters finales proyectados en 2D con PCA (solo visualización)")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout()
plt.show()

print(f"Varianza explicada por las 2 primeras componentes: {var_explicada.sum():.1%}")
print("(Baja cobertura esperable en 100 dimensiones: es una ayuda visual, no un resumen fiel de la matriz.)")


### Variables distintivas por cluster (heatmap de importancia estandarizada)

In [ ]:
# Top features numéricas más importantes en conjunto (unión de los top-6 de cada cluster)
top_por_cluster = (
    cluster_characterization[cluster_characterization["FEATURE"].isin(numericas)]
    .sort_values(["CLUSTER", "IMPORTANCE"], ascending=[True, False])
    .groupby("CLUSTER")
    .head(6)
)
features_heatmap = top_por_cluster["FEATURE"].unique().tolist()

pivot_diff = (
    cluster_characterization[cluster_characterization["FEATURE"].isin(features_heatmap)]
    .pivot(index="FEATURE", columns="CLUSTER", values="IMPORTANCE")
    .reindex(features_heatmap)
)

fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(features_heatmap))))
sns.heatmap(pivot_diff, cmap="viridis", annot=True, fmt=".2f", ax=ax, cbar_kws={"label": "|diferencia| estandarizada"})
ax.set_title("Variables numéricas más distintivas por cluster (efecto estandarizado vs. global)")
ax.set_xlabel("Cluster")
plt.tight_layout()
plt.show()


In [ ]:
# Comparación de perfiles: TIPOEMPLEADO_ACTUAL_DESC y VIGENTE_ACTUALMENTE por cluster
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ct_tipo = pd.crosstab(df_carac["CLUSTER"], df_carac["TIPOEMPLEADO_ACTUAL_DESC"], normalize="index")
ct_tipo.plot(kind="bar", stacked=True, ax=axes[0], color=["#DD8452", "#4C72B0"])
axes[0].set_title("Tipo de empleado por cluster")
axes[0].set_ylabel("proporción")
axes[0].legend(title=None, fontsize=8)

vigencia = df_carac.groupby("CLUSTER")["VIGENTE_ACTUALMENTE"].apply(lambda s: s.astype("boolean").mean(skipna=True))
axes[1].bar(vigencia.index.astype(str), vigencia.values, color="#55A868")
axes[1].axhline(df_carac["VIGENTE_ACTUALMENTE"].astype("boolean").mean(skipna=True), color="gray", linestyle="--", label="promedio global")
axes[1].set_title("Proporción actualmente vigente por cluster")
axes[1].set_ylabel("proporción vigente")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 10. Resumen

**Decisión tomada:** K-Means con **K=5**, `random_state=42`, `n_init=10`, ajustado sobre las 100
columnas de `X_modelado.csv`. Se descartó la partición trivial K=2 (mejor Silhouette pero sin
separación sustantiva), se validó cruzadamente con clustering jerárquico (ward), y se documentó que
DBSCAN/HDBSCAN no son adecuados para esta matriz de alta dimensionalidad con variables dummy dispersas.

**Archivos generados en `data/clustering/`:**

| Archivo | Contenido |
|---|---|
| `evaluacion_k.csv` | Métricas de K-Means para K=2..12 |
| `clusters_personas.csv` | `IDPERSONA` + `CLUSTER` final (una fila por persona) |
| `modelo_clustering.joblib` | Modelo K-Means final serializado |
| `clustering_metrics.csv` | Métricas del modelo final seleccionado |
| `cluster_sizes.csv` | Tamaño y % de población por cluster |
| `cluster_profiles.csv` | Resumen por cluster con features originales (media/mediana/proporciones/modas) |
| `cluster_characterization.csv` | Detalle `CLUSTER, FEATURE, VALUE_CLUSTER, VALUE_GLOBAL, DIFFERENCE, IMPORTANCE, INTERPRETACION` |

**Limitaciones y advertencias a tener presentes:**

- La matriz `X_modelado` mezcla variables numéricas estandarizadas con variables dummy 0/1 sin
  estandarizar; esto pondera más las variables numéricas en la distancia euclídea usada por K-Means
  y por el jerárquico.
- Los nombres de perfil de la sección 8 son una síntesis interpretativa, no una categoría
  institucional oficial ni una verdad absoluta sobre las personas — así lo establece el contexto
  permanente del proyecto.
- La proyección PCA de la sección 9 es solo una ayuda visual (baja varianza explicada en 2D dado que
  hay 100 dimensiones); no reemplaza ni resume fielmente la matriz usada para clustering.
- Esta solución no debe usarse para automatizar decisiones sensibles de contratación, promoción o
  asignación sin intervención humana.

**Siguiente paso sugerido:** registrar esta decisión (K=5, K-Means, criterios de selección) como una
entrada en `context/DECISION_LOG.md`, y continuar con `notebooks/06_embeddings/06_embeddings.ipynb`
para explorar si una representación semántica de texto complementa estos perfiles estructurados.
